# TrainLM: LLM Training & MFU Benchmark on Colab TPU v5e-1

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dhiraj309/TrainLM/blob/main/notebooks/TrainLM_v5e1_MFU_Benchmark.ipynb)

Standard Hugging Face UX — `TrainLMTrainer` handles BF16 placement, whole-step `torch.compile`, `DeferAndFuse` eager fallback, and `chunked_linear` loss automatically underneath:

```python
trainer = TrainLMTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=TrainLMTrainingArguments(
        output_dir="./out",
        per_device_train_batch_size=8,
        sequence_length=2048,
        max_steps=100,
        bf16=True,
        accelerator="native_tpu",
        torch_compile=True,
        loss_implementation="chunked_linear",
    ),
)
trainer.train()
```

The cells below benchmark **SmolLM2-135M**, **SmolLM2-360M**, **Qwen2.5-0.5B**, and **Qwen3-0.6B** (`head_dim=128`) on **Cloud TPU v5e-1** (16 GB HBM, 197 TFLOP/s BF16 peak), separating **Window 1 (cold compile)** from **Windows 2+ (steady-state MFU)**.

In [ ]:
import json
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, distribution, version

BRANCH = "main"  # Change to a PR branch if testing pre-merge.
REPO_URL = "https://github.com/Dhiraj309/TrainLM.git"

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", f"trainlm @ git+{REPO_URL}@{BRANCH}"],
    check=True,
)
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--quiet",
        "huggingface-hub>=1.5,<2",
        "PyYAML>=6,<7",
        "transformers>=5.15,<6",
        "datasets>=3,<5",
        "tokenizers>=0.21,<1",
    ],
    check=True,
)

direct_url = json.loads(distribution("trainlm").read_text("direct_url.json") or "{}")
SOURCE_REVISION = direct_url.get("vcs_info", {}).get("commit_id")
PACKAGES = {}
for pkg in ("trainlm", "torch", "torch-tpu", "libtpu", "transformers", "datasets"):
    try:
        PACKAGES[pkg] = version(pkg)
    except PackageNotFoundError:
        PACKAGES[pkg] = None

print("source_revision=", SOURCE_REVISION)
print("packages=", json.dumps(PACKAGES, indent=2))

In [ ]:
# Select which models and modes to benchmark at 2K sequence length on TPU v5e-1 (16 GB HBM) or GPU.
MODELS_TO_RUN = ("smollm2-135m", "smollm2-360m", "qwen2.5-0.5b", "qwen3-0.6b")  # Add "llama3.2-1b" for 1B+ tests
RUN_EAGER = True          # Measure eager baseline on SmolLM2-135M (and fallback if compile fails).
RUN_COMPILED = True       # Measure torch_compile=True on all selected models.
RUN_PROFILER = True       # Capture top 10 CPU/host ops on SmolLM2-135M eager.
STEPS = 25                # 25 steps / 5 logging_steps -> Window 1 (cold) + Windows 2-5 (steady-state).
LOGGING_STEPS = 5
USE_SYNTHETIC_DATA = True # True isolates compute/dispatch MFU; False streams TinyStories with varlen packing.
TORCH_COMPILE_MODE = "default"  # "default", "reduce-overhead", or "max-autotune-no-cudagraphs"
TPU_VM_MACHINE_TYPE = None


In [ ]:
import gc
import importlib.util
import tempfile
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from trainlm import TrainLMTrainer, TrainLMTrainingArguments
from trainlm.optimization import probe_torchtpu_kernels
from trainlm.kernels import is_pallas_ce_available, is_pallas_splash_available, is_triton_available
from trainlm.kernels.pallas.splash_attention import probe_pallas_bridge_spec
from trainlm.runtime import discover_tpu

MODEL_SPECS = {
    "smollm2-135m": {"model_id": "HuggingFaceTB/SmolLM2-135M", "seq_len": 2048, "batch_size": 8, "loss": "chunked_linear", "grad_ckpt": False},
    "smollm2-360m": {"model_id": "HuggingFaceTB/SmolLM2-360M", "seq_len": 2048, "batch_size": 4, "loss": "chunked_linear", "grad_ckpt": False},
    "qwen2.5-0.5b": {"model_id": "Qwen/Qwen2.5-0.5B", "seq_len": 2048, "batch_size": 2, "loss": "chunked_linear", "logits_chunk_size": 1024, "grad_ckpt": False},
    "qwen3-0.6b": {"model_id": "Qwen/Qwen3-0.6B", "seq_len": 2048, "batch_size": 2, "loss": "chunked_linear", "logits_chunk_size": 1024, "grad_ckpt": False},
    "llama3.2-1b": {"model_id": "unsloth/Llama-3.2-1B", "seq_len": 2048, "batch_size": 1, "loss": "chunked_linear", "grad_ckpt": True},
}
BENCHMARK_RESULTS = {}

TPU_DEVICE_NAME = ""
TPU_HBM_GB = 0.0
if importlib.util.find_spec("torch_tpu") is not None:
    import torch_tpu
    print("torch.tpu public attrs:", [x for x in dir(getattr(torch, "tpu", None)) if not x.startswith("_")])
    print("torch_tpu public attrs:", [x for x in dir(torch_tpu) if not x.startswith("_")])
    tpu_info = discover_tpu(torch, verify_allocation=False, include_properties=True)
    TPU_DEVICE_NAME = str(tpu_info.get("device_name") or "")
    TPU_HBM_GB = (tpu_info.get("hbm_bytes_per_device") or 0) / (1024 ** 3)
    print(f"Detected TPU: {TPU_DEVICE_NAME or 'unknown'} | HBM={TPU_HBM_GB:.1f} GiB")

IS_TPU_V6E = "V6E" in TPU_DEVICE_NAME.upper() or TPU_HBM_GB >= 30.0
if IS_TPU_V6E:
    MODEL_SPECS["smollm2-135m"]["batch_size"] = 16
    MODEL_SPECS["smollm2-360m"]["batch_size"] = 8
    MODEL_SPECS["qwen2.5-0.5b"]["batch_size"] = 8
    MODEL_SPECS["qwen3-0.6b"]["batch_size"] = 4
    MODEL_SPECS["llama3.2-1b"]["batch_size"] = 2
    print("torch.ops.tpu public attrs:", [x for x in dir(getattr(torch.ops, "tpu", None)) if not x.startswith("_")])


def run_model_benchmark(key, *, run_eager=False, run_compiled=True, profile_ops=False):
    spec = MODEL_SPECS[key]
    for mode, compile_enabled in (("eager", False), ("compiled", True)):
        if (mode == "eager" and not run_eager) or (mode == "compiled" and not run_compiled):
            continue
        label = f"{key}-{mode}"
        print(f"\n=== Running {label} ({spec['model_id']}, seq={spec['seq_len']}, bs={spec['batch_size']}) ===", flush=True)
        try:
            tokenizer = AutoTokenizer.from_pretrained(spec["model_id"])
            model = AutoModelForCausalLM.from_pretrained(spec["model_id"])
            if USE_SYNTHETIC_DATA:
                vocab = int(getattr(model.config, "vocab_size", len(tokenizer)))
                rows = max(64, spec["batch_size"] * STEPS)
                tokens = torch.randint(0, vocab, (rows, spec["seq_len"]), dtype=torch.long)
                dataset = [{"input_ids": r, "labels": r.clone()} for r in tokens]
            else:
                from datasets import load_dataset
                raw = load_dataset("roneneldan/TinyStories", split="train[:200]")
                if tokenizer.pad_token is None:
                    tokenizer.pad_token = tokenizer.eos_token
                dataset = raw.map(lambda b: tokenizer(b["text"], truncation=True, max_length=spec["seq_len"], padding=False), batched=True, remove_columns=raw.column_names)

            use_bf16 = (not torch.cuda.is_available()) or torch.cuda.is_bf16_supported()
            optim_choice = "adamw_bf16" if (use_bf16 and not torch.cuda.is_available() and importlib.util.find_spec("torch_tpu") is not None) else "adamw"
            trainer = TrainLMTrainer(
                model=model,
                processing_class=tokenizer,
                train_dataset=dataset,
                args=TrainLMTrainingArguments(
                    output_dir=tempfile.mkdtemp(prefix=f"trainlm-{label}-"),
                    max_steps=STEPS,
                    sequence_length=spec["seq_len"],
                    per_device_train_batch_size=spec["batch_size"],
                    learning_rate=1e-4,
                    logging_steps=LOGGING_STEPS,
                    eval_strategy="no",
                    save_strategy="no",
                    bf16=use_bf16,
                    fp16=not use_bf16,
                    accelerator="cuda" if torch.cuda.is_available() else ("native_tpu" if importlib.util.find_spec("torch_tpu") else "cpu"),
                    torch_compile=compile_enabled,
                    torch_compile_mode=TORCH_COMPILE_MODE,
                    gradient_checkpointing=spec["grad_ckpt"],
                    use_liger_kernel="auto",
                    use_fused_rms_norm=True,
                    pad_vocab_to_multiple="auto",
                    loss_rematerialization="disabled" if IS_TPU_V6E else "auto",
                    tpu_allow_excess_precision="auto",
                    tpu_compile_cache_dir="auto",
                    prefetch_batches=True,
                    optim=optim_choice,
                    packing=not USE_SYNTHETIC_DATA,
                    dataloader_pin_memory=torch.cuda.is_available(),
                    loss_implementation=spec["loss"],
                    logits_chunk_size=spec.get("logits_chunk_size", 2048) if spec["loss"] == "chunked_linear" else None,
                ),
            )
            liger_res = getattr(trainer, "_liger_result", None)
            rms_res = getattr(trainer, "_rms_norm_result", None)
            align_rep = getattr(trainer, "_alignment_report", None)
            tpu_inv = probe_torchtpu_kernels()
            bridge_spec = probe_pallas_bridge_spec()
            print(
                f"  opts: liger={bool(liger_res and liger_res.applied)} "
                f"fused_rms_norm={getattr(rms_res, 'patched_modules', 0)} "
                f"vocab_aligned={getattr(align_rep, 'vocab_aligned', None)} "
                f"splash={tpu_inv.has_splash_attention} pallas_bridge={bridge_spec[0] if bridge_spec else False} pallas_splash={is_pallas_splash_available()} pallas_ce={is_pallas_ce_available()} triton={is_triton_available()} optim={optim_choice}",
                flush=True,
            )
            trainer.train()
            perf = trainer.performance_summary()
            BENCHMARK_RESULTS[f"{key}_{mode}"] = perf
            stats = perf.get("steady_state") or perf
            print(
                f"[{label}] steady={stats.get('tokens_per_second', 0):.1f} tok/s "
                f"| MFU={((stats.get('non_embedding_mfu') or 0) * 100):.2f}% "
                f"(logits={((stats.get('logits_inclusive_mfu') or 0) * 100):.2f}%) "
                f"| wait={stats.get('dataloader_wait_fraction', 0):.1%}",
                flush=True,
            )
            if profile_ops and mode == "eager":
                with torch.profiler.profile(activities=[torch.profiler.ProfilerActivity.CPU]) as prof:
                    raw_batch = next(iter(trainer.engine.train_dataloader))
                    step_res = trainer.engine.task.training_step(trainer.model, raw_batch, trainer.runtime)
                    trainer.runtime.backward(step_res.loss)
                    trainer.runtime.optimizer_step(trainer.optimizer)
                    trainer.runtime.zero_grad(trainer.optimizer)
                    trainer.runtime.synchronize()
                print(prof.key_averages().table(sort_by="self_cpu_time_total", row_limit=10))
                del raw_batch, step_res, prof
            del trainer, model
        except Exception as exc:
            BENCHMARK_RESULTS[f"{key}_{mode}"] = {"status": "failed", "error": f"{type(exc).__name__}: {exc}"}
            print(f"[{label}] FAILED: {type(exc).__name__}: {exc}", flush=True)
        if hasattr(torch, "_dynamo") and hasattr(torch._dynamo, "reset"):
            try:
                torch._dynamo.reset()
            except Exception:
                pass
        gc.collect()
        if hasattr(torch, "accelerator") and hasattr(torch.accelerator, "empty_cache"):
            try:
                torch.accelerator.empty_cache()
            except Exception:
                pass
        elif torch.cuda.is_available():
            torch.cuda.empty_cache()


### 1. SmolLM2-135M (Eager + Compiled)

In [ ]:
if "smollm2-135m" in MODELS_TO_RUN:
    run_model_benchmark("smollm2-135m", run_eager=RUN_EAGER, run_compiled=RUN_COMPILED, profile_ops=RUN_PROFILER)

### 2. SmolLM2-360M (Compiled + `chunked_linear`)

In [ ]:
if "smollm2-360m" in MODELS_TO_RUN:
    run_model_benchmark("smollm2-360m", run_eager=(RUN_EAGER and not RUN_COMPILED), run_compiled=RUN_COMPILED)

### 3. Qwen2.5-0.5B (Compiled + `chunked_linear`)

In [ ]:
if "qwen2.5-0.5b" in MODELS_TO_RUN:
    run_model_benchmark("qwen2.5-0.5b", run_eager=(RUN_EAGER and not RUN_COMPILED), run_compiled=RUN_COMPILED)

### 4. Qwen3-0.6B (Compiled + `head_dim=128` + Pallas Splash & CE)


In [ ]:
if "qwen3-0.6b" in MODELS_TO_RUN:
    run_model_benchmark("qwen3-0.6b", run_eager=(RUN_EAGER and not RUN_COMPILED), run_compiled=RUN_COMPILED)


In [ ]:
print("\n=== MFU Benchmark Summary ===")
for name, perf in BENCHMARK_RESULTS.items():
    if "error" in perf:
        print(f"  {name:24s} | FAILED: {perf['error']}")
    else:
        stats = perf.get("steady_state") or perf
        mfu = (stats.get("non_embedding_mfu") or 0) * 100
        l_mfu = (stats.get("logits_inclusive_mfu") or 0) * 100
        print(f"  {name:24s} | {stats.get('tokens_per_second', 0):8.1f} tok/s | MFU={mfu:5.2f}% (logits={l_mfu:5.2f}%) | wait={stats.get('dataloader_wait_fraction', 0):.1%}")

REPORT = {
    "schema_version": 1,
    "source_revision": SOURCE_REVISION,
    "packages": PACKAGES,
    "use_synthetic_data": USE_SYNTHETIC_DATA,
    "results": BENCHMARK_RESULTS,
}
REPORT_PATH = Path.cwd() / "trainlm_v5e1_mfu_benchmark.json"
REPORT_PATH.write_text(json.dumps(REPORT, indent=2, sort_keys=True, default=str) + "\n", encoding="utf-8")
print(f"Saved {REPORT_PATH}")
try:
    from google.colab import files
except ImportError:
    from IPython.display import FileLink, display
    display(FileLink(REPORT_PATH.name))
else:
    files.download(str(REPORT_PATH))